In [1]:
print(123)

123


In [2]:
from ingest import load_faq_data
documents = load_faq_data()

In [3]:
documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

len(documents_llm)

103

In [4]:
documents = documents_llm

In [5]:
doc = documents[0]
print(doc["id"])
print(doc["question"])
print(doc["answer"])

74eb249bbf
I just discovered the course. Can I still join?
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


In [6]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
data_gen_instructions = """
You emulate a student who's taking our course.
Formulate 5 questions this student might ask based on a FAQ record. The record
should contain the answer to the questions, and the questions should be complete and not too short.
If possible, use as fewer words as possible from the record.

The output should resemble how people ask questions
on the internet. Not too formal, not too short, not too long.
""".strip()

In [8]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [9]:
import json

user_prompt = json.dumps(doc)

In [10]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [11]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [12]:
result = response.output_parsed

print(result)

questions=['I just found this course — can I still enroll and start now?', 'Is it too late to join the course if I missed the beginning?', 'Can new students still participate in the course at this point?', 'If I join late, will I still be able to get a certificate?', 'What’s the deadline for submitting the project if I want the certificate?']


In [13]:
print(result.questions)

['I just found this course — can I still enroll and start now?', 'Is it too late to join the course if I missed the beginning?', 'Can new students still participate in the course at this point?', 'If I join late, will I still be able to get a certificate?', 'What’s the deadline for submitting the project if I want the certificate?']


In [14]:
from evaluation_utils import llm_structured

In [15]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['I just found this course — can I still sign up and follow along, or is it too late to join?', 'Am I allowed to start the course now even though it’s already underway?', 'If I join late, is there anything special I need to do to still get a certificate?', 'Do I need to finish and submit the project before submissions close in order to earn the certificate?', 'Can I still participate in the course after it started, and will I only get the certificate if I submit the project on time?']


In [16]:
usage.input_tokens, usage.output_tokens

(207, 118)

In [17]:
from evaluation_utils import calc_price

In [18]:
cost = calc_price(usage)

cost

{'input_cost': 0.00015525, 'output_cost': 0.000531, 'total_cost': 0.00068625}

In [19]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["id"]
    })

records

[{'question': 'I just found this course — can I still sign up and follow along, or is it too late to join?',
  'document': '74eb249bbf'},
 {'question': 'Am I allowed to start the course now even though it’s already underway?',
  'document': '74eb249bbf'},
 {'question': 'If I join late, is there anything special I need to do to still get a certificate?',
  'document': '74eb249bbf'},
 {'question': 'Do I need to finish and submit the project before submissions close in order to earn the certificate?',
  'document': '74eb249bbf'},
 {'question': 'Can I still participate in the course after it started, and will I only get the certificate if I submit the project on time?',
  'document': '74eb249bbf'}]

In [20]:
from evaluation_utils import llm_structured_retry

In [21]:
def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["id"]
        })

    return results, usage

In [22]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(documents[:5]):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/5 [00:00<?, ?it/s]

In [23]:
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

In [24]:
with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, documents, generate_ground_truth)

  0%|          | 0/103 [00:00<?, ?it/s]

In [25]:
ground_truth = []
usages = []

for records, usage in results:
    ground_truth.extend(records)
    usages.append(usage)

len(ground_truth)

515

In [26]:
from evaluation_utils import calc_price

total_cost = 0.0

for usage in usages:
    cost = calc_price(usage)
    total_cost = total_cost + cost["total_cost"]

total_cost

0.07746825000000002

In [27]:
from evaluation_utils import calc_total_price

calc_total_price(usages)

0.07746825000000002

In [28]:
import pandas as pd

df_ground_truth = pd.DataFrame(ground_truth)

In [30]:
df_ground_truth.to_csv("data/ground_truth-new.csv", index=False)